# `Ratings.csv` vs `goodreads_books.json` — coverage, descriptions, genre`archive/Ratings.csv` holds the user-book interactions (`User-ID`, `ISBN`,`Book-Rating`). This notebook answers:1. How many distinct books are in `Ratings.csv`?2. What share of them appear in the 8.6 GB Goodreads dump?3. Of those, how many have a usable **description**?4. Is there a **genre** feature anywhere in the dump?`goodreads_books.json` is **JSON Lines**, not a JSON array — one complete object perline — so everything below streams the file and never loads it into memory.

In [ ]:
import json
import pickle
import time
from collections import Counter
from pathlib import Path

import pandas as pd

DATA = Path("/Users/jakubhajko/Projects/bookshelf")
BOOKS_JSON = DATA / "goodreads_books.json"
RATINGS_CSV = DATA / "archive" / "Ratings.csv"
CACHE = DATA / ".feature_cache.pkl"

## 1. One example record from the big JSONEvery scalar is a **string** (`"average_rating": "4.00"`), missing values are `""`not `null`, and there are two ISBN fields: `isbn` (10-digit) and `isbn13`.

In [ ]:
with open(BOOKS_JSON, "r", encoding="utf-8") as f:
    example = json.loads(next(f))

print(json.dumps(example, indent=2, ensure_ascii=False))

## 2. Unique books in `Ratings.csv`

In [ ]:
def norm(s):
    # ISBNs in the wild carry hyphens, spaces and lowercase 'x' check digits.
    return str(s).strip().upper().replace("-", "").replace(" ", "")


ratings = pd.read_csv(RATINGS_CSV)
ratings["isbn_n"] = ratings["ISBN"].map(norm)
bx_unique = set(ratings["isbn_n"].unique())

print(f"rating rows          : {len(ratings):,}")
print(f"unique users         : {ratings['User-ID'].nunique():,}")
print(f"unique books (ISBNs) : {len(bx_unique):,}")

## 3. One streaming pass over the 8.6 GBTo answer *any* coverage question we need to look at all ~2.36M records, not asample. So we make a **single pass** (~40 s) that collects everything at once:- `desc10` / `desc13` — maps ISBN → **length of that book's description**. The keys  double as the set of ISBNs present; the values answer the description question, so  one pass covers both.- `shelf_books` — how many books carry each `popular_shelves` tag (for the genre  question in §5).- `allkeys` — every field name seen anywhere, to check whether a genre field exists.Cached to `.feature_cache.pkl`; delete that file to force a rescan.

In [ ]:
def scan(path):
    desc10, desc13 = {}, {}
    shelf_books = Counter()
    allkeys = Counter()
    n = 0
    t = time.perf_counter()
    with open(path, "rb") as f:
        for line in f:
            r = json.loads(line)
            allkeys.update(r.keys())
            L = len((r["description"] or "").strip())
            if r["isbn"]:
                k = norm(r["isbn"])
                if L > desc10.get(k, -1):
                    desc10[k] = L
            if r["isbn13"]:
                k = norm(r["isbn13"])
                if L > desc13.get(k, -1):
                    desc13[k] = L
            for sh in r["popular_shelves"]:
                shelf_books[sh["name"]] += 1
            n += 1
            if n % 400_000 == 0:
                print(f"  {n:,} records  {time.perf_counter()-t:.0f}s", flush=True)
    print(f"scanned {n:,} records in {time.perf_counter()-t:.0f}s")
    # trim the shelf tail so the cache stays small
    shelf_books = {k: v for k, v in shelf_books.items() if v >= 50}
    return {"desc10": desc10, "desc13": desc13, "shelf_books": shelf_books,
            "allkeys": dict(allkeys), "n_records": n}


if CACHE.exists():
    with open(CACHE, "rb") as f:
        S = pickle.load(f)
    print(f"loaded from cache: {CACHE.name}")
else:
    S = scan(BOOKS_JSON)
    with open(CACHE, "wb") as f:
        pickle.dump(S, f, protocol=5)

desc10, desc13 = S["desc10"], S["desc13"]
n_records = S["n_records"]
print(f"\ngoodreads records : {n_records:,}")
print(f"unique isbn10     : {len(desc10):,}")
print(f"unique isbn13     : {len(desc13):,}")

## 4. The overlapBook-Crossing uses **ISBN-10**, so the primary join is Goodreads' `isbn` field. A fewGoodreads records carry only an `isbn13`, so leftovers are converted 10 → 13 andretried. Note one book can be reached by either route, so we resolve eachBook-Crossing ISBN to a single description length.

In [ ]:
def isbn10_to_13(i10):
    core = i10[:9]
    if not core.isdigit():
        return None
    body = "978" + core
    tot = sum(int(d) * (1 if k % 2 == 0 else 3) for k, d in enumerate(body))
    return body + str((10 - tot % 10) % 10)


direct = {i for i in bx_unique if i in desc10}
extra = {i for i in bx_unique - direct
         if (c := isbn10_to_13(i)) and c in desc13}
matched = direct | extra

# bx isbn -> description length, whichever route found it
desc_len = {i: desc10[i] for i in direct}
desc_len.update({i: desc13[isbn10_to_13(i)] for i in extra})

pd.DataFrame([
    ("unique books in Ratings.csv", len(bx_unique), 1.0),
    ("matched on isbn10 (direct)", len(direct), len(direct) / len(bx_unique)),
    ("matched after 10->13 convert", len(extra), len(extra) / len(bx_unique)),
    ("MATCHED TOTAL", len(matched), len(matched) / len(bx_unique)),
    ("not in goodreads", len(bx_unique) - len(matched),
     1 - len(matched) / len(bx_unique)),
], columns=["", "books", "share"]).style.format(
    {"books": "{:,}", "share": "{:.1%}"}).hide(axis="index")

In [ ]:
covered = ratings["isbn_n"].isin(matched)
print(f"unique books covered : {len(matched):,} / {len(bx_unique):,} "
      f"= {len(matched)/len(bx_unique):.1%}")
print(f"rating ROWS covered  : {covered.sum():,} / {len(ratings):,} "
      f"= {covered.mean():.1%}")
print("\nThe gap is the point: the books that match are the popular ones,")
print("so they carry far more of the ratings than their title count suggests.")

## 5. How many matched books have a description?`description` is always *present* as a key but is very often the empty string, so"has a description" means non-empty after stripping.

In [ ]:
dl = pd.Series(desc_len)
n_m = len(matched)

rows = [("matched books", n_m, 1.0)]
rows.append(("non-empty description", int((dl > 0).sum()), (dl > 0).mean()))
for thr in (100, 200, 500, 1000):
    k = int((dl >= thr).sum())
    rows.append((f"description >= {thr:,} chars", k, k / n_m))
rows.append(("empty description", int((dl == 0).sum()), (dl == 0).mean()))

pd.DataFrame(rows, columns=["", "books", "share of matched"]).style.format(
    {"books": "{:,}", "share of matched": "{:.1%}"}).hide(axis="index")

In [ ]:
print("description length, non-empty only (characters):")
print(dl[dl > 0].describe(percentiles=[.25, .5, .75, .9]).round(0).to_string())

has_desc = {k for k, v in desc_len.items() if v > 0}
cov_desc = ratings["isbn_n"].isin(has_desc)
print(f"\nrating rows whose book is matched AND has a description: "
      f"{cov_desc.sum():,} ({cov_desc.mean():.1%})")

## 6. Genre**There is no genre field in this file.** All 29 keys below appear in every one of the~2.36M records, and none is a genre, category, subject or tag field — verified overthe whole file, not a sample.

In [ ]:
ak = pd.Series(S["allkeys"]).sort_values(ascending=False)
print(f"{len(ak)} distinct keys; all present in {ak.iloc[0]:,} / {n_records:,} records")
print(sorted(ak.index))

hits = [k for k in ak.index
        if any(s in k.lower() for s in ("genre", "categ", "subject", "tag", "topic"))]
print(f"\ngenre-like field names: {hits or 'NONE'}")

### What to use instead: `popular_shelves``popular_shelves` is the de-facto genre signal — the user-applied tags for each book,with counts. It is free text, so it mixes real genres (`fiction`, `fantasy`,`mystery`) with shelving verbs (`to-read`, `currently-reading`, `owned`) and personaljunk (`read-in-2016`, `home-library`). Filtering the obvious non-genres gets you ausable label set.This is exactly how the official genre labels were built: the UCSD release's`goodreads_book_genres_initial.json.gz` (23 MB) is described as *"genre tags extractedfrom users' popular shelves by a simple keyword matching process"* — so if you wantready-made genres, download that file and join on `book_id` rather than reinventingthe keyword matching. It collapses shelves into ~10 coarse labels(`fiction`, `fantasy, paranormal`, `mystery, thriller, crime`, `romance`, `history,historical fiction, biography`, `children`, `young-adult`, `comics, graphic`,`poetry`, `non-fiction`).

In [ ]:
NON_GENRE = {
    "to-read", "currently-reading", "default", "owned", "books-i-own", "owned-books",
    "favorites", "favourites", "favorite", "my-books", "library", "to-buy", "wishlist",
    "wish-list", "kindle", "audiobook", "audiobooks", "audio", "ebook", "ebooks",
    "e-book", "e-books", "books", "book", "series", "school", "re-read", "reread",
    "did-not-finish", "didn-t-finish", "dnf", "abandoned", "unfinished", "unread",
    "my-library", "have", "i-own", "all-books", "paperback", "hardcover", "bookshelf",
    "tbr", "own-it", "own", "home-library", "book-club", "english", "shelfari-favorites",
    "maybe", "reviewed", "audible", "wanted", "general", "stand-alone", "standalone",
}
# The list above is not exhaustive — popular_shelves is free text and always leaves
# some noise. Extend it as you spot more shelving verbs in the output.

shelves = pd.Series(S["shelf_books"], name="n_books")
genreish = shelves[~shelves.index.isin(NON_GENRE)]
genreish = genreish[~genreish.index.str.match(r"^(read|owned)?-?in-?\d{4}$")]

top = genreish.sort_values(ascending=False).head(25).to_frame()
top["% of all books"] = (top.n_books / n_records * 100).round(1)
top

In [ ]:
# Shelves on the example record, highest count first — this is what you'd distill.
ex = sorted(example["popular_shelves"], key=lambda s: -int(s["count"]))
pd.DataFrame(ex).head(10)